## Read Healsparse

In [ ]:
import healsparse
import healpy as hp
import matplotlib.pyplot as plt
from pathlib import Path
import os
import re

SPMAP_DIR = "SPMAP_DATADIR"
path = Path(SPMAP_DIR)

In [ ]:
band_order = {'u': 0, 'g': 1, 'r': 2, 'i': 3, 'z': 4, 'y': 5}
def sort_by_band(filename):
    # On cherche quelle lettre de band_order est présente dans le nom du fichier
    for band in band_order:
        # On vérifie "_u_" ou "u_band" ou juste "u" selon votre nomenclature
        # Ici on cherche la bande isolée (ex: _u. ou _u_)
        if f'_{band}' in filename or f' {band} ' in filename:
            return band_order[band]
    return 99  # Au cas où une bande n'est pas reconnue

In [ ]:
all_map_files = os.listdir(SPMAP_DIR)

In [ ]:
all_map_files

In [ ]:
# 3. Tri de la liste
sorted_files = sorted(all_map_files, key=sort_by_band)

print(sorted_files)

In [ ]:
# Récupérer uniquement les fichiers FITS

path = Path(SPMAP_DIR)
files = list(path.glob("*.fits*"))

# Trier les objets Path
files.sort(key=lambda f: band_order.get(re.search(r'[_]([ugrizy])[_]', f.name).group(1), 99))

for f in files:
    print(f"Fichier prêt : {f.name}")

In [ ]:
files

In [ ]:
map_filename = files[1]

In [ ]:
# Rechargement
hsp = healsparse.HealSparseMap.read(map_filename)
# Vérification
print(type(hsp))
print(f"Pixels valides : {hsp.n_valid}")

In [ ]:
# 1. Convertir en carte Healpix standard (dense)
# Note : nside=128 ou 256 suffit souvent pour une vue globale

hsp_dense = hsp.generate_healpix_map(nside=32)

# 2. Affichage Mollweide standard
hp.mollview(hsp_dense, 
            title=os.path.basename(map_filename),
            cmap="viridis",
            unit = "psf_maglim",
            norm="hist") # ou norm=None pour linéaire

hp.graticule(dpar=30, dmer=30, color='white', alpha=0.5, lw=0.5)
# 3. Ajouter les labels de Latitude (Parallèles)
# On écrit les chiffres le long du méridien central
for lat in [-60, -30, 0, 30, 60]:
    hp.projtext(0, lat, f'{lat}°', lonlat=True, coord='C', color='black', fontsize=10)

# 4. Ajouter les labels de Longitude (Méridiens)
# On écrit les chiffres le long de l'équateur
for lon in [0, 60, 120, 180, 240, 300]:
    # On décale un peu en latitude pour ne pas chevaucher la ligne
    hp.projtext(lon, 5, f'{lon}°', lonlat=True, coord='C', color='black', fontsize=10)

plt.show()